In [1]:
import json
import os
import numpy as np

cluster_path = os.path.join(os.getcwd(), "data_for_git", "all_query_clusters.jsonl")
all_query_clusters = []
with open(cluster_path, "r") as f:
    for line in f:
        all_query_clusters.append(json.loads(line))

matrices_path = os.path.join(os.getcwd(), "data", "matrices.npy")
all_query_matrices = np.load(matrices_path, allow_pickle=True)


responses_path = os.path.join(os.getcwd(), "data_for_git", "responses.jsonl")
all_query_responses = []
with open(responses_path, "r") as f:
    for line in f:
        all_query_responses.append(json.loads(line))



In [ ]:
import numpy as np

def calculate_blackbox_degree_uncertainty(
    clusters_list,       # Your list of cluster objects
    entailment_matrix,   # Matrix/Dict where [child_idx][parent_idx] = 1 (or score)
):
    """
    Calculates Degree Uncertainty (U_deg) and Confidence (C_deg).
    
    Logic:
    1. Identify all N sequences (by UUID).
    2. Build a 'Semantic Set' for each sequence.
       - If Sequence A contains Cluster 5 ('Nobel'), add 5 to Set A.
       - EXPANSION: If Cluster 5 implies Cluster 20 ('Award'), add 20 to Set A too.
    3. Compute Jaccard Similarity between sets.
    """
    
    # --- Step 1: Map Sequence UUIDs to Indices 0..m ---
    # We need a stable ordering for the similarity matrix
    seq_uuids = set()
    for cluster in clusters_list:
        for atom in cluster["atomic_facts"]:
            seq_uuids.add(atom["from_sequence"])
            
    # Sort for deterministic behavior
    sorted_uuids = sorted(list(seq_uuids))
    uuid_to_idx = {uuid: i for i, uuid in enumerate(sorted_uuids)}
    
    actual_num_seqs = len(sorted_uuids)
    if actual_num_seqs == 0:
        return {"uncertainty": 1.0, "confidence": []}

    # --- Step 2: Build Semantic Sets ---
    # sets[i] = {set of cluster indices present in sequence i}
    seq_sets = [set() for _ in range(actual_num_seqs)]
    
    # Pre-calculate entailment expansions (Implied Facts)
    # If Cluster i -> Cluster j, then presence of i implies j.
    # We assume entailment_matrix matches the list order of 'clusters_list'
    num_clusters = len(clusters_list)
    implied_map = {i: set() for i in range(num_clusters)}

    for child in range(num_clusters):
        seen = set()
        stack = [child]
        while stack:
            u = stack.pop()
            for v in range(num_clusters):
                if u == v:
                    continue
                if entailment_matrix[u][v][2] > 0.5 and v not in seen:
                    seen.add(v)
                    implied_map[child].add(v)
                    stack.append(v)

    # Populate Sets
    for c_idx, cluster in enumerate(clusters_list):
        # 1. Identify which sequences contain this cluster
        # We look at 'atomic_facts' to see who contributed
        participating_seqs = set()
        for atom in cluster["atomic_facts"]:
            s_uuid = atom["from_sequence"]
            if s_uuid in uuid_to_idx:
                participating_seqs.add(uuid_to_idx[s_uuid])
        
        for s_idx in participating_seqs:
            seq_sets[s_idx].add(c_idx)
            for implied_idx in implied_map[c_idx]:
                seq_sets[s_idx].add(implied_idx)

    # --- Step 3: Jaccard Similarity Matrix ---
    W = np.zeros((actual_num_seqs, actual_num_seqs))
    
    for i in range(actual_num_seqs):
        for j in range(actual_num_seqs):
            set_i = seq_sets[i]
            set_j = seq_sets[j]
            
            # Edge Case: Empty sequences (Hallucination or Silence)
            if len(set_i) == 0 and len(set_j) == 0:
                W[i, j] = 1.0 # Agreement on silence
            elif len(set_i) == 0 or len(set_j) == 0:
                W[i, j] = 0.0 # Disagreement
            else:
                intersect = len(set_i.intersection(set_j))
                union = len(set_i.union(set_j))
                W[i, j] = intersect / union

    # --- Step 4: Degree & Uncertainty ---
    # Degree = Sum of similarities (Row Sum)
    degrees = np.sum(W, axis=1)
    
    # Confidence[i] = Degree[i] / m
    confidence_scores = degrees / actual_num_seqs
    
    # Uncertainty = 1 - Mean(Confidence)
    # (Matches trace formula: Trace(mI - D)/m^2)
    uncertainty_score = 1.0 - np.mean(confidence_scores)
    
    return {
        "uncertainty": uncertainty_score,
        "confidence": confidence_scores.tolist(),
        "degrees": degrees.tolist()
    }

In [ ]:
for i in range(len(all_query_matrices)):
    clusters = all_query_clusters[i]["clusters"]
    blackbox_result = calculate_blackbox_degree_uncertainty(clusters, all_query_matrices[i])


In [4]:
print(blackbox_result)


{'uncertainty': np.float64(0.4986666666666666), 'confidence': [0.4966666666666667, 0.5199999999999999, 0.44000000000000006, 0.4833333333333333, 0.5666666666666667], 'degrees': [2.4833333333333334, 2.5999999999999996, 2.2, 2.4166666666666665, 2.833333333333333]}
